# GroupDNA – WhatsApp Group Analyzer

**Name:** Devipriya S  
**Roll Number:** 17997

**Batch:** Data science July 2026 Batch  
**Date:** 25 July 2026

In [26]:
with open('/content/DADS Minor PROJECT dataset.txt','r', encoding='UTF-8') as f:
  contents=f.read()
lines=contents.split("\n")

# Feature 1 – Chat Parser


In [27]:
# AI-assisted: Used AI to understand WhatsApp chat parsing logic and handling of system, media and deleted messages.
chatt_records=[]
system_messagess=0
media_messages=0
deleted_messages=0
for line in lines:
  if line=="":
    continue
  time_content,message_content=line.split("-",1)

  date,time=time_content.split(",")
  if ":" not in message_content:
    system_messagess=system_messagess+1
    continue
  sender, message=message_content.split(":",1)
  if message.strip()=="<Media omitted>":
    media_messages=media_messages+1
    continue
  if message.strip()=="This message was deleted":
    deleted_messages=deleted_messages+1
    continue

  records={"time":time,
           "date":date,
         "sender":sender,
         "messages":message}

  chatt_records.append(records)
  group_members=set()

  for records in chatt_records:
    group_members.add(records["sender"])

print("Messages Parsed :", len(chatt_records))
print("Participants    :", len(group_members))
print("System Messages :", system_messagess)
print("Media Messages  :", media_messages)
print("Deleted Messages:", deleted_messages)


Messages Parsed : 3127
Participants    : 6
System Messages : 4
Media Messages  : 32
Deleted Messages: 15



# Feature 2 – Group Overview

In [41]:
# AI-assisted: Used AI to improve formatting and calculate group statistics.

from datetime import datetime

# Count messages per participant
messages_per_participant = {}

for record in chatt_records:
    sender = record["sender"]

    if sender in messages_per_participant:
        messages_per_participant[sender] += 1
    else:
        messages_per_participant[sender] = 1

# Calculate chat duration
total_messages = len(chatt_records)

start_date = datetime.strptime(
    chatt_records[0]["date"].strip(),
    "%d/%m/%y"
)

end_date = datetime.strptime(
    chatt_records[-1]["date"].strip(),
    "%d/%m/%y"
)

days = (end_date - start_date).days + 1

average_messages_per_day = total_messages / days

# Print Group Overview
print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)

print(f"Group           :Hostel Bois 4ever")
print(
    f"Period          : {start_date.strftime('%d %B %Y')} to "
    f"{end_date.strftime('%d %B %Y')} ({days} days)"
)
print(f"Total Messages  : {total_messages}")
print(f"Participants    : {len(group_members)}")

print("\nMESSAGES PER PERSON")

sorted_members = sorted(
    messages_per_participant.items(),
    key=lambda x: x[1],
    reverse=True
)

for person, count in sorted_members:
    percentage = (count / total_messages) * 100
    print(f"{person:<10}: {count:<5} ({percentage:.1f}%)")

print(f"\nAverage Messages Per Day : {average_messages_per_day:.1f}")

GROUP OVERVIEW
Group           :Hostel Bois 4ever
Period          : 01 April 2024 to 30 May 2024 (60 days)
Total Messages  : 3127
Participants    : 6

MESSAGES PER PERSON
 Rahul    : 940   (30.1%)
 Priya    : 712   (22.8%)
 Neha     : 624   (20.0%)
 Aman     : 484   (15.5%)
 Karan    : 345   (11.0%)
 Vikas    : 22    (0.7%)

Average Messages Per Day : 52.1


# Feature 3 – Most Active Day and Hour



In [31]:
#Feature 3 - Most Active Day and Hour
from datetime import datetime

# -------------------- Busiest Day --------------------

day_count = {}

for record in chatt_records:
    day = record["date"].strip()

    if day in day_count:
        day_count[day] += 1
    else:
        day_count[day] = 1

busiest_day = max(day_count, key=day_count.get)
busiest_day_messages = day_count[busiest_day]

formatted_day = datetime.strptime(
    busiest_day,
    "%d/%m/%y"
).strftime("%d %B %Y")


# -------------------- Busiest Hour --------------------

hour_count = {}

for record in chatt_records:
    hour = int(record["time"].strip().split(":")[0])

    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

busiest_hour = max(hour_count, key=hour_count.get)
busiest_hour_messages = hour_count[busiest_hour]


print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)

print(f"Busiest day  : {formatted_day} ({busiest_day_messages} messages)")

print(
    f"Busiest hour : "
    f"{busiest_hour:02}:00 - {(busiest_hour + 1) % 24:02}:00 "
    f"({busiest_hour_messages} messages)"
)
days = (end_date - start_date).days + 1

average_hour = busiest_hour_messages / days

print(
    f"Busiest hour : "
    f"{busiest_hour:02}:00 - {(busiest_hour + 1) % 24:02}:00 "
    f"(avg {average_hour:.1f} messages per day)"
)

MOST ACTIVE DAY AND HOUR
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00 (244 messages)
Busiest hour : 18:00 - 19:00 (avg 4.1 messages per day)


# Feature 4 – Activity Heatmap (NumPy)


In [32]:
 # AI-assisted: Used AI to understand NumPy matrix creation and text-based heatmap rendering.
import numpy as np
group_members=sorted(list(group_members))
participant_index={}

for i,person in enumerate(group_members):
  participant_index[person]=i
heatmap = np.zeros((len(group_members),24),dtype=int)

for record in chatt_records:
  sender=record['sender']
  hour=int(record['time'].split(":")[0])
  row=participant_index[sender]
  heatmap[row][hour]+=1



print("ACTIVITY HEATMAP (messages by hour)")

# Print every 3rd hour
print("         ", end="")
for h in range(0, 24, 3):
    print(f"{h:02}", end=" ")
print()

for i, person in enumerate(group_members):

    print(f"{person:<8}", end=" ")

    maximum = np.max(heatmap[i])
    if maximum == 0:
        maximum = 1

    for h in range(0, 24, 3):

        value = heatmap[i][h]
        ratio = value / maximum

        if ratio == 0:
            block = "."
        elif ratio <= 0.25:
            block = "░"
        elif ratio <= 0.50:
            block = "▒"
        elif ratio <= 0.75:
            block = "▓"
        else:
            block = "█"

        print(block, end="  ")

    print()

ACTIVITY HEATMAP (messages by hour)
         00 03 06 09 12 15 18 21 
 Aman    ▓  ▓  .  .  .  ░  ░  ░  
 Karan   .  .  .  ▒  █  ▓  ▓  ▒  
 Neha    .  .  ░  █  ▓  ░  █  ▒  
 Priya   .  .  ░  █  █  ▒  ▓  ▒  
 Rahul   ░  ░  ░  ░  ▓  ▓  █  █  
 Vikas   .  .  .  ▒  ▒  ▒  ▓  ▒  


# Feature 5 – Top Words

In [33]:
# AI-assisted: Used AI to improve word frequency counting and stop-word filtering.
stop_words = {
    "i","me","my","we","our","you","your","he","she","it","they",
    "is","am","are","was","were","be","been","being",
    "the","a","an","and","or","to","of","yeah", "okay", "ok", "yes", "no","in","on","for","at","by",
    "with","this","that","these","those","as","from","if","then",
    "so","how","what","when","where","why","who","which",
    "have","everyone","one","up","started","entire","please","telling","has","had","do","does","did","will","would","can","could",
    "today","tomorrow","just","about","his","her","their","there",
    "okay","ok","yes","no","hai"
}

In [34]:
word_count = {}

for record in chatt_records:
    message = record["messages"].lower()
    words = message.split()
    for word in words:
        word = word.strip(".,!?;:()[]{}\"'<>")
        if word == "":
            continue
        if word in stop_words:
            continue
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

In [35]:
top_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)

In [36]:

print("=" * 50)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 50)

for word, count in top_words[:10]:
  maximum = top_words[0][1]
  bar = "█" * int((count / maximum) * 20)
  print(f"{word:<12} {bar:<25} {count}")

THIS GROUP'S FAVOURITE WORDS
guys         ████████████████████      318
bhai         ██████████                160
scene        █████████                 145
anyone       ████████                  139
yaar         ████████                  139
but          ████████                  138
kya          ████████                  133
now          ███████                   121
everything   ███████                   121
came         ███████                   116


# Feature 6 – Response Speed and Silent Streaks


In [37]:
# AI-assisted: Used AI to understand datetime parsing and response time calculation.
from datetime import datetime
response_time = {}

for person in group_members:
    response_time[person] = []

for i in range(1, len(chatt_records)):

    current = chatt_records[i]
    previous = chatt_records[i-1]

    if current["sender"] != previous["sender"]:

        previous_time = datetime.strptime(
           previous["date"].strip() + " " + previous["time"].strip(),
            "%d/%m/%y %H:%M" )

        current_time = datetime.strptime(
           current["date"].strip() + " " + current["time"].strip(),
            "%d/%m/%y %H:%M"
        )

        gap = (current_time - previous_time).total_seconds()

        response_time[current["sender"]].append(gap)
average_response = {}
for person, time in sorted(
    average_response.items(),
    key=lambda x: x[1]
):
    print(f"{person:<10}{time/60:.2f} minutes")

for person in response_time:

    if len(response_time[person]) > 0:

        average_response[person] = (
            sum(response_time[person])
            /
            len(response_time[person])
        )
print("="*50)
print("AVERAGE RESPONSE TIME")
print("="*50)

for person in average_response:

    minutes = average_response[person] / 60

    print(f"{person:<10}{minutes:.2f} minutes")
fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)

print()
print("Fastest Replier :", fastest)
print("Slowest Replier :", slowest)


AVERAGE RESPONSE TIME
 Aman     54.91 minutes
 Karan    36.83 minutes
 Neha     41.30 minutes
 Priya    42.57 minutes
 Rahul    35.17 minutes
 Vikas    34.90 minutes

Fastest Replier :  Vikas
Slowest Replier :  Aman


# Feature 7 – Personality Archetypes

In [38]:
# AI-assisted: Used AI to refine archetype assignment based on messaging behaviour.
spammer = max(messages_per_participant, key=messages_per_participant.get)
night_messages = {}

for person in group_members:
    night_messages[person] = 0

for record in chatt_records:
    hour = int(record["time"].strip().split(":")[0])

    if hour >= 23 or hour <= 5:
        night_messages[record["sender"]] += 1

night_owl = max(night_messages, key=night_messages.get)
morning_messages = {}

for person in group_members:
    morning_messages[person] = 0

for record in chatt_records:
    hour = int(record["time"].strip().split(":")[0])

    if 5 <= hour < 9:
        morning_messages[record["sender"]] += 1

early_bird = max(morning_messages, key=morning_messages.get)
fastest_replier = fastest

print("=" * 50)
print("GROUP PERSONALITY ARCHETYPES")
print("=" * 50)

print(f"Spammer           : {spammer}")
print(f"Night Owl         : {night_owl}")
print(f"Early Bird        : {early_bird}")
print(f"Fastest Replier   : {fastest_replier}")


GROUP PERSONALITY ARCHETYPES
Spammer           :  Rahul
Night Owl         :  Aman
Early Bird        :  Priya
Fastest Replier   :  Vikas


# Feature 8 – Final Report


In [39]:
# AI-assisted: Used AI to improve formatting and presentation of the final report.
print("=" * 70)
print("                 GROUPDNA FINAL REPORT")
print("=" * 70)

print("\nGROUP OVERVIEW")
print("-" * 70)
print(f"Total Messages        : {len(chatt_records)}")
print(f"Participants          : {len(group_members)}")
print(f"Most Active Member    : {most_active}")
print(f"Messages per Day      : {avg_messages_per_day:.2f}")
print(f"Chat Duration        : {chat_duration}")
print(f"Most Active Day     : {formatted_day}")
print(f"Most Active Hour    : {busiest_hour:02}:00 - {(busiest_hour+1)%24:02}:00")

print("\nTOP 10 WORDS")
print("-" * 70)

maximum = top_words[0][1]

for word, count in top_words[:10]:
    bar = "█" * int((count / maximum) * 20)
    print(f"{word:<15}{bar:<20} {count}")

print("\nRESPONSE ANALYSIS")
print("-" * 70)

for person in average_response:
    print(f"{person:<10}{average_response[person]/60:.2f} minutes")

print()
print(f"Fastest Replier : {fastest}")
print(f"Slowest Replier : {slowest}")

print("\nGROUP PERSONALITY")
print("-" * 70)
print(f"Spammer         : {spammer}")
print(f"Night Owl       : {night_owl}")
print(f"Early Bird      : {early_bird}")
print(f"Fastest Replier : {fastest_replier}")

print("\nMONTHLY ACTIVITY")
print("-" * 70)

for month, count in messages_per_month.items():
    print(f"{month:<20}{count}")

print("\n" + "=" * 70)
print("                 END OF GROUPDNA REPORT")
print("=" * 70)

                 GROUPDNA FINAL REPORT

GROUP OVERVIEW
----------------------------------------------------------------------
Total Messages        : 3127
Participants          : 6
Most Active Member    :  Rahul
Messages per Day      : 53.00
Chat Duration        : 59 days, 22:14:00
Most Active Day     : 04 May 2024
Most Active Hour    : 18:00 - 19:00

TOP 10 WORDS
----------------------------------------------------------------------
guys           ████████████████████ 318
bhai           ██████████           160
scene          █████████            145
anyone         ████████             139
yaar           ████████             139
but            ████████             138
kya            ████████             133
now            ███████              121
everything     ███████              121
came           ███████              116

RESPONSE ANALYSIS
----------------------------------------------------------------------
 Aman     54.91 minutes
 Karan    36.83 minutes
 Neha     41.30 minutes


# Reflection

Initially, I found this project a little difficult because there were many parts to combine into one program. The response time calculation and the heatmap took me the longest to complete because I kept getting small errors and had to debug them.

By the end of the project, I became much more comfortable with Python concepts like dictionaries, loops, file handling, string manipulation, datetime, and NumPy. It also improved my problem-solving skills.

If I continue working on this project, I would like to add more personality archetypes and improve the overall report. I also want to make it work with real WhatsApp chats that contain multiline messages.
